### OSM EXTRACTION FOR ARCHETYPE CLASSIFICATION

In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

In [2]:
# SETUP & FETCH STATION COORDINATES FROM OSM
ox.settings.log_console = True

# Query OSM for railway stations tagged as part of the SRT Red Line (Dark/Light) - search by station name within the Bangkok/Pathum Thani area
station_names_th = {
    'TLC': 'Taling Chan',
    'BMR': 'Bang Bamru',
    'BSN': 'Bang Son',
    'KTW': 'Krung Thep Aphiwat',
    'CTK': 'Chatuchak',
    'WSN': 'Wat Samian Nari',
    'BKH': 'Bang Khen',
    'TSH': 'Thung Song Hong',
    'LAK': 'Lak Si',
    'KHA': 'Kan Kheha',
    'DMG': 'Don Mueang',
    'LHK': 'Lak Hok',
    'RST': 'Rangsit',
}

# Fetch railway station points via Overpass
tags = {'railway': 'station'}
gdf = ox.features_from_place('Bangkok, Thailand', tags)
print(gdf.shape)
print(gdf[['name', 'geometry']].head(20))

(167, 69)
                                              name                    geometry
element id                                                                    
node    70753469                            สามเสน  POINT (100.52994 13.77944)
        257066779                        สุรศักดิ์  POINT (100.52146 13.71927)
        257066790                        ช่องนนทรี   POINT (100.52932 13.7238)
        257066809                          ราชดำริ  POINT (100.53945 13.73947)
        257067085                      สะพานตากสิน  POINT (100.51422 13.71881)
        259723396                          จิตรลดา  POINT (100.52471 13.76654)
        280179404                         หัวลำโพง  POINT (100.51714 13.73757)
        280179427                         พระราม 9  POINT (100.56528 13.75782)
        280179448  ศูนย์การประชุมแห่งชาติสิริกิติ์  POINT (100.55991 13.72267)
        280222462                          ลุมพินี  POINT (100.54572 13.72561)
        280239009                         

In [3]:
# CHECK ALTERNATE NAME COLUMNS
print([c for c in gdf.columns if 'name' in c.lower()])

['name', 'name:en', 'name:th', 'name:de', 'name:fr', 'name:zh', 'short_name', 'short_name:en', 'short_name:th', 'name:ja', 'int_name', 'name:ko', 'alt_name:en', 'name:ru', 'alt_name', 'loc_name', 'loc_name:de', 'loc_name:en', 'loc_name:th', 'loc_name:zh', 'name:es', 'alt_name:th', 'name:ka', 'official_name', 'official_name:en', 'official_name:th']


In [4]:
# FILTER USING name:en / int_name COLUMNS
red_line_keywords = ['Taling Chan', 'Bang Bamru', 'Bang Son', 'Krung Thep Aphiwat', 'Chatuchak', 'Wat Samian Nari', 'Bang Khen', 'Thung Song Hong', 'Lak Si', 'Kan Kheha', 'Don Mueang', 'Lak Hok', 'Rangsit', 'Bang Sue']

name_col = gdf['name:en'].astype(str)
intname_col = gdf['int_name'].astype(str)

mask = (name_col.str.contains('|'.join(red_line_keywords), case=False, na=False) | intname_col.str.contains('|'.join(red_line_keywords), case=False, na=False))

red_line_stations = gdf[mask][['name', 'name:en', 'int_name', 'geometry']]
print(red_line_stations)
print(f"\nFound {len(red_line_stations)} stations")
# 8 of our 13 stations found directly: BSN, CTK, WSN, BKH, TSH, LAK, KHA, KTW. Missing: TLC, BMR, DMG, LHK, RST

                                name               name:en int_name  \
element id                                                            
node    280240585         สวนจตุจักร        Chatuchak Park      NaN   
        1724588735     ชุมทางบางซื่อ     Bang Sue Junction      NaN   
        3641278609    ชุมทางตลิ่งชัน  Taling Chan Junction      NaN   
        5222817656           บางซ่อน              Bang Son      NaN   
        7787093471           จตุจักร             Chatuchak      NaN   
        7787093472     วัดเสมียนนารี       Wat Samian Nari      NaN   
        7787093474            บางเขน             Bang Khen      NaN   
        7787093475       ทุ่งสองห้อง       Thung Song Hong      NaN   
        7787093476           หลักสี่                Lak Si      NaN   
        7787093477           การเคหะ             Kan Kheha      NaN   
        8986366369   กรุงเทพอภิวัฒน์    Krung Thep Aphiwat      NaN   
        8990382534           บางซ่อน              Bang Son      NaN   
      

In [5]:
# BROADER SEARCH INCLUDING PATHUM THANI, AND HALT TAG
tags2 = {'railway': ['station', 'halt']}
gdf2 = ox.features_from_place('Pathum Thani, Thailand', tags2)

name_col2 = gdf2['name:en'].astype(str) if 'name:en' in gdf2.columns else pd.Series('', index=gdf2.index)
mask2 = name_col2.str.contains('|'.join(red_line_keywords), case=False, na=False)
print(gdf2[mask2][['name', 'name:en', 'geometry']] if 'name:en' in gdf2.columns else gdf2[mask2][['name','geometry']])
print(f"\nFound {mask2.sum()} in Pathum Thani")
# RST and LHK found.

                                 name               name:en  \
element id                                                    
node    6024782068             รังสิต               Rangsit   
        7787093479  หลักหก (ม.รังสิต)  Lak Hok (Rangsit U.)   

                                      geometry  
element id                                      
node    6024782068  POINT (100.60215 13.99054)  
        7787093479  POINT (100.60538 13.96572)  

Found 2 in Pathum Thani


In [6]:
# SEARCH BANGKOK FOR ALL railway=station/halt/stop, CHECK THAI NAMES TOO
tags_all = {'railway': ['station', 'halt', 'stop']}
gdf4 = ox.features_from_place('Bangkok, Thailand', tags_all)

# Thai names for Taling Chan, Bang Bamru, Don Mueang
thai_keywords = ['ตลิ่งชัน', 'บางบำหรุ', 'ดอนเมือง']
name_th = gdf4['name'].astype(str)
mask4 = name_th.str.contains('|'.join(thai_keywords), case=False, na=False)
print(gdf4[mask4][['name', 'geometry']])
print(f"\nFound {mask4.sum()}")
# TLC, BMR and DMG found.

                               name                    geometry
element id                                                     
node    259729383          บางบำหรุ  POINT (100.47754 13.79199)
        259729384    ชุมทางตลิ่งชัน  POINT (100.43993 13.78913)
        2039197506   ชุมทางตลิ่งชัน  POINT (100.44009 13.78929)
        3641278609   ชุมทางตลิ่งชัน  POINT (100.44008 13.78927)
        3641278704   ชุมทางตลิ่งชัน  POINT (100.44001 13.78919)
        3641278705   ชุมทางตลิ่งชัน  POINT (100.44007 13.78926)
        3641278710   ชุมทางตลิ่งชัน  POINT (100.44016 13.78935)
        7314656147         ดอนเมือง  POINT (100.59799 13.91462)
        7314656148         ดอนเมือง  POINT (100.59798 13.91463)
        7314656149         ดอนเมือง  POINT (100.59789 13.91468)
        7314656150         ดอนเมือง  POINT (100.59788 13.91468)
        7314656151         ดอนเมือง   POINT (100.59785 13.9147)
        7314656152         ดอนเมือง   POINT (100.59785 13.9147)
        7314656153         ดอนเมือง  POI

In [7]:
# CONSOLIDATE FINAL 13-STATION COORDINATE TABLE
station_coords = {
    'TLC': (13.78927, 100.44008),   # Taling Chan Junction
    'BMR': (13.79199, 100.47754),   # Bang Bamru
    'BSN': (13.82008, 100.53246),   # Bang Son (node 5222817656)
    'KTW': (13.80403, 100.54171),   # Krung Thep Aphiwat
    'CTK': (13.80296, 100.55328),   # Chatuchak (Park) - node 280240585
    'WSN': (13.84159, 100.55750),   # Wat Samian Nari
    'BKH': (13.84937, 100.56167),   # Bang Khen
    'TSH': (13.86021, 100.56753),   # Thung Song Hong
    'LAK': (13.88631, 100.58194),   # Lak Si
    'KHA': (13.89866, 100.58887),   # Kan Kheha
    'DMG': (13.91469, 100.59787),   # Don Mueang
    'LHK': (13.96572, 100.60538),   # Lak Hok
    'RST': (13.99054, 100.60215),   # Rangsit
}

coords_df = pd.DataFrame.from_dict(station_coords, orient='index', columns=['lat','lon'])
coords_df.index.name = 'station'
print(coords_df)

# QUICK SANITY CHECK: STATIONS SHOULD BE ROUGHLY ORDERED NORTH-EASTWARD ALONG THE LINE
print("\nSorted by latitude (should roughly follow line order TLC->RST):")
print(coords_df.sort_values('lat'))

              lat        lon
station                     
TLC      13.78927  100.44008
BMR      13.79199  100.47754
BSN      13.82008  100.53246
KTW      13.80403  100.54171
CTK      13.80296  100.55328
WSN      13.84159  100.55750
BKH      13.84937  100.56167
TSH      13.86021  100.56753
LAK      13.88631  100.58194
KHA      13.89866  100.58887
DMG      13.91469  100.59787
LHK      13.96572  100.60538
RST      13.99054  100.60215

Sorted by latitude (should roughly follow line order TLC->RST):
              lat        lon
station                     
TLC      13.78927  100.44008
BMR      13.79199  100.47754
CTK      13.80296  100.55328
KTW      13.80403  100.54171
BSN      13.82008  100.53246
WSN      13.84159  100.55750
BKH      13.84937  100.56167
TSH      13.86021  100.56753
LAK      13.88631  100.58194
KHA      13.89866  100.58887
DMG      13.91469  100.59787
LHK      13.96572  100.60538
RST      13.99054  100.60215


In [6]:
# EXTRACT 1KM CATCHMENT FEATURES FOR EACH STATION
import warnings
warnings.filterwarnings('ignore')

catchment_radius = 1000  # meters

# Tags of interest: land use, buildings, POIs
poi_tags = {
    'amenity': True,
    'shop': True,
    'leisure': True,
    'tourism': True,
}
landuse_tags = {'landuse': True}
building_tags = {'building': True}

catchment_summary = []

for station, row in coords_df.iterrows():
    lat, lon = row['lat'], row['lon']
    print(f"Processing {station}...")
    try:
        pois = ox.features_from_point((lat, lon), tags=poi_tags, dist=catchment_radius)
        buildings = ox.features_from_point((lat, lon), tags=building_tags, dist=catchment_radius)
        landuse = ox.features_from_point((lat, lon), tags=landuse_tags, dist=catchment_radius)

        catchment_summary.append({
            'station': station,
            'n_pois': len(pois),
            'n_buildings': len(buildings),
            'n_landuse_polygons': len(landuse),
        })
    except Exception as e:
        print(f"  Error for {station}: {e}")
        catchment_summary.append({'station': station, 'n_pois': 0, 'n_buildings': 0, 'n_landuse_polygons': 0})

catchment_df = pd.DataFrame(catchment_summary)
print(catchment_df)

Processing TLC...
Processing BMR...
Processing BSN...
Processing KTW...
Processing CTK...
Processing WSN...
Processing BKH...
Processing TSH...
Processing LAK...
  Error for LAK: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x1803f1be210>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))
Processing KHA...
  Error for KHA: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x1803f1bead0>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))
Processing DMG...
  Error for DMG: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x1803f1befd0>, 'Conn

In [ ]:
# OVERPASS API ENDPOINTS
# https://overpass.kumi.systems/api/interpreter #migrated to Private.coffee
# https://overpass.private.coffee/api/interpreter
# https://overpass-api.de/api/interpreter
# https://maps.mail.ru/osm/tools/overpass/api/interpreter

In [16]:
# SET ALTERNATIVE OVERPASS ENDPOINT, TEST WITH TLC ONLY
import osmnx as ox
ox.settings.overpass_url = "https://overpass-api.de/api/interpreter"
ox.settings.requests_timeout = 180

catchment_radius = 1000  # meters
poi_tags = {
    'amenity': True,
    'shop': True,
    'leisure': True,
    'tourism': True,
}

lat, lon = coords_df.loc['TLC', 'lat'], coords_df.loc['TLC', 'lon']
print("Testing TLC...")
pois = ox.features_from_point((lat, lon), tags=poi_tags, dist=catchment_radius)
print(f"TLC: pois={len(pois)}")

Testing TLC...
TLC: pois=22


In [17]:
# RETRY FAILED STATIONS (LAK, KHA, DMG, LHK, RST)
# SWITCH TO ALTERNATE OVERPASS MIRROR AND RETRY
import time
ox.settings.overpass_endpoint = "https://overpass-api.de/api/interpreter"
ox.settings.requests_timeout = 180

failed_stations = ['LAK', 'KHA', 'DMG', 'LHK', 'RST']
retry_results = []

for station in failed_stations:
    row = coords_df.loc[station]
    lat, lon = row['lat'], row['lon']
    print(f"Retrying {station}...")
    try:
        pois = ox.features_from_point((lat, lon), tags=poi_tags, dist=catchment_radius)
        time.sleep(2)
        buildings = ox.features_from_point((lat, lon), tags=building_tags, dist=catchment_radius)
        time.sleep(2)
        landuse = ox.features_from_point((lat, lon), tags=landuse_tags, dist=catchment_radius)
        time.sleep(2)

        retry_results.append({
            'station': station,
            'n_pois': len(pois),
            'n_buildings': len(buildings),
            'n_landuse_polygons': len(landuse),
        })
        print(f"  Success: pois={len(pois)}, buildings={len(buildings)}, landuse={len(landuse)}")
    except Exception as e:
        print(f"  Error for {station}: {e}")
        retry_results.append({'station': station, 'n_pois': None, 'n_buildings': None, 'n_landuse_polygons': None})

retry_df = pd.DataFrame(retry_results)
print(retry_df)

Retrying LAK...
  Success: pois=97, buildings=1045, landuse=44
Retrying KHA...
  Success: pois=57, buildings=691, landuse=17
Retrying DMG...
  Success: pois=261, buildings=153, landuse=29
Retrying LHK...
  Success: pois=11, buildings=118, landuse=7
Retrying RST...
  Success: pois=40, buildings=30, landuse=18
  station  n_pois  n_buildings  n_landuse_polygons
0     LAK      97         1045                  44
1     KHA      57          691                  17
2     DMG     261          153                  29
3     LHK      11          118                   7
4     RST      40           30                  18


In [11]:
# CHECK FOR UNIVERSITY/EDUCATION POI TAGS NEAR LHK (AND RST FOR COMPARISON)
import osmnx as ox
ox.settings.overpass_url = "https://overpass-api.de/api/interpreter"
ox.settings.requests_timeout = 180

catchment_radius = 1000  # meters
poi_tags = {
    'amenity': True,
    'shop': True,
    'leisure': True,
    'tourism': True,
}

for station in ['LHK', 'RST']:
    row = coords_df.loc[station]
    lat, lon = row['lat'], row['lon']
    pois = ox.features_from_point((lat, lon), tags=poi_tags, dist=catchment_radius)
    print(f"\n{station} POIs:")
    cols_to_check = [c for c in ['amenity','shop','leisure','tourism','name'] if c in pois.columns]
    print(pois[cols_to_check].to_string())


LHK POIs:
                              amenity    shop        leisure    tourism                                            name
element id                                                                                                             
node    3658633281   place_of_worship     NaN            NaN        NaN                                       วัดรังสิต
        4650839709                NaN     NaN            NaN  viewpoint                                             NaN
        6054062361   place_of_worship     NaN            NaN        NaN                                             NaN
        11765981506               NaN  beauty            NaN        NaN                                  Bonnie Eyelash
way     30738633                  NaN     NaN    golf_course        NaN                               สนามกอล์ฟเมืองเอก
        43856917                  NaN     NaN         common        NaN                                             NaN
        691059207            

___
**END**